# PDDL Copilot — Experiment Runner

Interactive reproduction of the evaluation from [Benyamin et al., 2025 (arXiv:2509.12987)](https://arxiv.org/abs/2509.12987).

Evaluates Ollama LLMs **with** and **without** MCP planning tools on PDDL tasks.

## Configuration

Set the path to your pddl-copilot marketplace clone and choose models/tasks to evaluate.

In [ ]:
# --- Configuration (edit these) ---
MARKETPLACE_PATH = "/path/to/pddl-copilot"  # <-- set this to your clone
MODELS = ["qwen3:0.5b", "qwen3:4b"]
TASKS = ["solve", "validate_domain", "validate_problem", "validate_plan", "simulate"]
NUM_VARIANTS = 5
TEMPERATURE = 0.0
RUN_CHAINS = False
CHAIN_SAMPLES = 20
SEED = 42

In [ ]:
import random
from pathlib import Path

from run_experiment import (
    MCPPlanner,
    load_domains,
    resolve_plugin_dirs,
    generate_ground_truth,
    run_single_task_experiment,
    run_chain_experiment,
    print_single_task_table,
    print_chain_table,
    save_results,
    RESULTS_DIR,
    DOMAINS_DIR,
)

random.seed(SEED)
plugin_dirs = resolve_plugin_dirs(MARKETPLACE_PATH)
print(f"Plugins: {[p.name for p in plugin_dirs]}")

## Load Domains

In [ ]:
domains = load_domains(DOMAINS_DIR)
n_problems = sum(len(d["problems"]) for d in domains.values())
print(f"Loaded {len(domains)} domains, {n_problems} problems total")
for dname, dinfo in domains.items():
    print(f"  {dname} ({dinfo['type']}): {len(dinfo['problems'])} problems")

## Connect MCP Servers

In [ ]:
mcp = MCPPlanner()
await mcp.connect(plugin_dirs)
print(f"\nAvailable tools: {[t['function']['name'] for t in mcp.tools]}")

## Generate Ground Truth

Solve all problems using MCP planners as oracle (Section 4.2 of the paper).

In [ ]:
ground_truth = await generate_ground_truth(mcp, domains)

## Run Single-Task Evaluation

In [ ]:
single_results = await run_single_task_experiment(
    models=MODELS,
    tasks=TASKS,
    domains=domains,
    ground_truth=ground_truth,
    mcp=mcp,
    num_variants=NUM_VARIANTS,
)
print_single_task_table(single_results)

## Run Multi-Task Chain Evaluation (Optional)

Set `RUN_CHAINS = True` in the configuration cell above to enable this.

In [ ]:
chain_results = []
if RUN_CHAINS:
    chain_results = await run_chain_experiment(
        models=MODELS,
        domains=domains,
        ground_truth=ground_truth,
        mcp=mcp,
        chain_lengths=[2, 3, 4, 5],
        samples=CHAIN_SAMPLES,
    )
    print_chain_table(chain_results)
else:
    print("Chain evaluation skipped. Set RUN_CHAINS = True to enable.")

## Save Results

In [ ]:
save_results(single_results, chain_results, RESULTS_DIR)
print("Done!")

## Cleanup

In [ ]:
await mcp.close()
print("MCP connections closed.")